# House Price Prediction Using Linear Regression
## Technical demonstration and portfolio project

**Data provenance warning:** This notebook uses a synthetically generated dataset of 180 records. It is not sourced from property transactions, listings, an estate agency, or a government database. Therefore, the model is a learning/portfolio demonstration and must not be used to quote, advertise, lend against, or make property purchase decisions.

**Target:** illustrative house price in million PKR.  
**Model:** ordinary least-squares Linear Regression.  
**Split:** 80% training / 20% test, with `random_state=42`.


## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

ROOT = Path.cwd()
for folder in ["data", "figures", "outputs", "model"]:
    (ROOT / folder).mkdir(exist_ok=True)


## 2. Load the supplied synthetic dataset

In [ ]:
df = pd.read_csv(ROOT / "data" / "house_prices_synthetic.csv")
print("Dataset shape:", df.shape)
display(df.head())


## 3. Data quality checks

In [ ]:
print(df.info())
print("\\nMissing values by column:")
display(df.isna().sum().to_frame("missing_values"))
print("Duplicate rows:", df.duplicated().sum())
display(df.describe().T)


## 4. Exploratory data analysis

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["price_million_pkr"], bins=15, edgecolor="black")
plt.title("Synthetic House Price Distribution")
plt.xlabel("Price (million PKR)")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df["area_sqft"], df["price_million_pkr"], alpha=0.75)
plt.title("Area vs Synthetic House Price")
plt.xlabel("Area (sq ft)")
plt.ylabel("Price (million PKR)")
plt.tight_layout()
plt.show()


## 5. Define features and target

In [ ]:
FEATURES = ["area_sqft", "bedrooms", "bathrooms", "house_age_years",
            "location_score", "parking_spaces"]
TARGET = "price_million_pkr"
X = df[FEATURES]
y = df[TARGET]
print("X shape:", X.shape, "| y shape:", y.shape)


## 6. Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))


## 7. Train Linear Regression

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
print("Intercept:", model.intercept_)
display(pd.DataFrame({"feature": FEATURES, "coefficient": model.coef_}))


## 8. Predict and evaluate

In [ ]:
y_pred = model.predict(X_test)
metrics = {
    "MAE (million PKR)": mean_absolute_error(y_test, y_pred),
    "MSE ((million PKR)^2)": mean_squared_error(y_test, y_pred),
    "RMSE (million PKR)": mean_squared_error(y_test, y_pred) ** 0.5,
    "R2": r2_score(y_test, y_pred),
}
display(pd.DataFrame(metrics.items(), columns=["metric", "value"]))


In [ ]:
predictions = pd.DataFrame({
    "actual_price_million_pkr": y_test.to_numpy(),
    "predicted_price_million_pkr": y_pred
})
predictions["absolute_error_million_pkr"] = (
    predictions["actual_price_million_pkr"] - predictions["predicted_price_million_pkr"]
).abs()
display(predictions.head(10))


## 9. Diagnostic plots

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.8)
limits = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(limits, limits, "r--", label="Ideal prediction")
plt.xlabel("Actual price (million PKR)")
plt.ylabel("Predicted price (million PKR)")
plt.title("Actual vs Predicted — Synthetic Test Set")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
residuals = y_test.to_numpy() - y_pred
plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.8)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted price (million PKR)")
plt.ylabel("Residual (actual - predicted)")
plt.title("Residual Analysis")
plt.tight_layout()
plt.show()


## 10. Model interpretation — permutation importance

In [ ]:
importance = permutation_importance(
    model, X_test, y_test, n_repeats=20, random_state=42, scoring="r2"
)
importance_df = pd.DataFrame({
    "feature": FEATURES,
    "mean_r2_decrease": importance.importances_mean,
    "std_r2_decrease": importance.importances_std
}).sort_values("mean_r2_decrease", ascending=False)
display(importance_df)


## 11. Save outputs and trained model

In [ ]:
ROOT.joinpath("outputs").mkdir(exist_ok=True)
pd.DataFrame(metrics.items(), columns=["metric", "value"]).to_csv(
    ROOT / "outputs" / "model_evaluation.csv", index=False
)
predictions.to_csv(ROOT / "outputs" / "test_set_predictions.csv", index=False)
importance_df.to_csv(ROOT / "outputs" / "permutation_importance.csv", index=False)
joblib.dump(model, ROOT / "model" / "linear_regression_model.pkl")
print("Metrics, predictions, feature importance, and model saved.")


## 12. Example prediction — illustrative only

In [ ]:
example_house = pd.DataFrame({
    "area_sqft": [2000],
    "bedrooms": [3],
    "bathrooms": [2],
    "house_age_years": [5],
    "location_score": [8],
    "parking_spaces": [1],
})
example_price = model.predict(example_house)[0]
print(f"Illustrative model output: {example_price:.2f} million PKR")
print("This is not a real-market valuation; the model was trained on synthetic data.")


## 13. Limitations and production-readiness checklist

This is not a validated real-estate valuation product. Before any real client deployment:
1. Obtain a properly licensed, representative dataset of actual property transactions or verified listings.
2. Define geography, currency, property type, valuation date, and intended use.
3. Audit missingness, duplicates, outliers, data leakage, and measurement definitions.
4. Create a time-aware and/or geographic holdout test where appropriate.
5. Compare a baseline with regularized and nonlinear models; tune only within training data.
6. Evaluate errors by location, property type, and price band; check calibration and drift.
7. Document data rights, privacy, consent, retention, and client acceptance criteria.
8. Obtain independent domain review before any commercial valuation use.

**Interpretation:** the reported R² is the proportion of target variation explained on this particular synthetic test split. It is not “97% accuracy” and does not establish performance on real houses.
